## Filters and Aggregates with `PySpark` and `SQL`

### Filter High-Order Values

In [0]:
%sql

-- filter by orders that have an AOC greater than 50 USD
SELECT order_id, city, state, cuisine, order_value_usd
FROM food_delivery_lab.bronze.orders
WHERE order_value_usd > 50
  AND delivery_status = 'Delivered';

In [0]:
from pyspark.sql.functions import col

df = spark.table("food_delivery_lab.bronze.orders")

filtered_df = df.filter(
    (col("order_value_usd") > 50) &
    (col("delivery_status") == "Delivered")
)

display(filtered_df)

### Filter Orders by Multiple Cities

In [0]:
%sql

-- select all orders in New York, Los Angeles and Chicago
SELECT *
FROM food_delivery_lab.bronze.orders
WHERE city IN ('New York', 'Los Angeles', 'Chicago');

In [0]:
city_df = df.filter(col("city").isin("New York", "Los Angeles", "Chicago"))

display(city_df)

### Handle Null Filtering

In [0]:
%sql

-- select order row where delivery_time is NULL
SELECT order_id, delivery_status, delivery_time
FROM food_delivery_lab.bronze.orders
WHERE delivery_time IS NULL;

In [0]:
non_na_df = df.filter(col("delivery_time").isNull())

display(non_na_df)

### Aggregate Revenue by City

In [0]:
%sql

-- aggregate revenue by city and arrange in decreasing order
SELECT city,
       SUM(order_value_usd) AS total_revenue
FROM food_delivery_lab.bronze.orders
WHERE delivery_status = 'Delivered'
GROUP BY city
ORDER BY total_revenue DESC;

In [0]:
from pyspark.sql.functions import sum

revenue_df = df.filter(col("delivery_status") == "Delivered") \
  .groupBy("city") \
  .agg(sum("order_value_usd").alias("total_revenue")) \
  .orderBy(col("total_revenue").desc())

display(revenue_df)

### Count Orders by Cuisine

In [0]:
%sql

-- count total orders grouped by cuisine and arranged in descending order
SELECT cuisine,
       COUNT(*) AS total_orders
FROM food_delivery_lab.bronze.orders
GROUP BY cuisine
ORDER BY total_orders DESC;

In [0]:
from pyspark.sql.functions import count

cuisine_df = df.groupBy("cuisine") \
  .agg(count("*").alias("total_orders")) \
  .orderBy(col("total_orders").desc())

display(cuisine_df)

### Commplex Aggregation - `Revenue` and `Average Order Value (AOV)`

In [0]:
%sql

SELECT city,
       COUNT(*) AS total_orders,
       SUM(order_value_usd) AS total_revenue,
       AVG(order_value_usd) AS avg_order_value
FROM food_delivery_lab.bronze.orders
GROUP BY city
ORDER BY total_revenue DESC;

In [0]:
from pyspark.sql.functions import count, sum, avg

complex_df = df.groupBy("city").agg(
    count("*").alias("total_orders"),
    sum("order_value_usd").alias("total_revenue"),
    avg("order_value_usd").alias("avg_order_value")
).orderBy(col("total_revenue").desc())

display(complex_df)